In [1]:
import numpy as np
import pandas as pd
import pickle

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

I0000 00:00:1790259375.613788   37159 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790259376.832278   37159 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790259381.338279   37159 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
df = pd.read_csv("Dataset/PRSA_Data_Guanyuan_20130301-20170228.csv")
print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(35064, 18)


In [5]:
print("Columns:")
print(df.columns.tolist())

Columns:
['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station']


In [6]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
No            0
year          0
month         0
day           0
hour          0
PM2.5       616
PM10        429
SO2         474
NO2         659
CO         1753
O3         1173
TEMP         20
PRES         20
DEWP         20
RAIN         20
wd           81
WSPM         14
station       0
dtype: int64


In [ ]:
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,datetime
0,1,2013,3,1,0,4.0,4.0,14.0,20.0,300.0,69.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Guanyuan,2013-03-01 00:00:00
1,2,2013,3,1,1,4.0,4.0,13.0,17.0,300.0,72.0,-1.1,1023.2,-18.2,0.0,N,4.7,Guanyuan,2013-03-01 01:00:00
2,3,2013,3,1,2,3.0,3.0,10.0,19.0,300.0,69.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Guanyuan,2013-03-01 02:00:00
3,4,2013,3,1,3,3.0,6.0,7.0,24.0,400.0,62.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Guanyuan,2013-03-01 03:00:00
4,5,2013,3,1,4,3.0,6.0,5.0,14.0,400.0,71.0,-2.0,1025.2,-19.5,0.0,N,2.0,Guanyuan,2013-03-01 04:00:00


In [ ]:
df = df[["datetime", "PM2.5"]]
df.head()

,datetime,PM2.5
0,2013-03-01 00:00:00,4.0
1,2013-03-01 01:00:00,4.0
2,2013-03-01 02:00:00,3.0
3,2013-03-01 03:00:00,3.0
4,2013-03-01 04:00:00,3.0


In [ ]:
df = df.set_index("datetime")
df = df.sort_index()
df.head()

,PM2.5
datetime,
2013-03-01 00:00:00,4.0
2013-03-01 01:00:00,4.0
2013-03-01 02:00:00,3.0
2013-03-01 03:00:00,3.0
2013-03-01 04:00:00,3.0


In [ ]:
df["PM2.5"] = pd.to_numeric(df["PM2.5"], errors="coerce")
print(df["PM2.5"].dtype)

float64


In [ ]:
df["PM2.5"] = df["PM2.5"].interpolate(method="time")
df = df.dropna()
print("Remaining Missing Values:")
print(df.isnull().sum())

Remaining Missing Values:
PM2.5    0
dtype: int64


In [ ]:
print("First Date:", df.index.min())
print("Last Date:", df.index.max())
print("Total Records:", len(df))

First Date: 2013-03-01 00:00:00
Last Date: 2017-02-28 23:00:00
Total Records: 35064


In [ ]:
data = df["PM2.5"].values.reshape(-1, 1)
print("Data Shape:", data.shape)

Data Shape: (35064, 1)


In [ ]:
train_size = int(len(data) * 0.80)
train_data = data[:train_size]
test_data = data[train_size:]
print("Training Samples:", len(train_data))
print("Testing Samples:", len(test_data))

Training Samples: 28051
Testing Samples: 7013


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

In [34]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [ ]:
SEQUENCE_LENGTH = 24
def create_sequences(data, sequence_length):
    X = []
    y = []
    for i in range(sequence_length, len(data)):
        X.append(data[i - sequence_length : i])
        y.append(data[i])
    return (np.array(X), np.array(y))

In [ ]:
X_train, y_train = create_sequences(train_scaled, SEQUENCE_LENGTH)
print("X_train Shape:", X_train.shape)
print("y_train Shape:", y_train.shape)

X_train Shape: (28027, 24, 1)
y_train Shape: (28027, 1)


In [ ]:
combined_data = np.concatenate((train_scaled[-SEQUENCE_LENGTH:], test_scaled), axis=0)
X_test, y_test = create_sequences(combined_data, SEQUENCE_LENGTH)
print("X_test Shape:", X_test.shape)
print("y_test Shape:", y_test.shape)

X_test Shape: (7013, 24, 1)
y_test Shape: (7013, 1)


In [ ]:
model = Sequential(
    [
        LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, 1)),
        LSTM(32),
        Dense(1),
    ]
)
model.summary()

E0000 00:00:1790260005.737659   37159 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/anas/LEARNING/GEN-AI/python/venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,345 (114.63 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer="adam", loss="mean_squared_error")

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=1,
)

Epoch 1/30
789/789 ━━━━━━━━━━━━━━━━━━━━ 28s 28ms/step - loss: 0.0020 - val_loss: 7.5097e-04
Epoch 2/30
789/789 ━━━━━━━━━━━━━━━━━━━━ 16s 21ms/step - loss: 8.0032e-04 - val_loss: 5.9939e-04
Epoch 3/30
789/789 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - loss: 7.6402e-04 - val_loss: 7.1253e-04
Epoch 4/30
789/789 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - loss: 7.5572e-04 - val_loss: 6.0938e-04
Epoch 5/30
789/789 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - loss: 7.5278e-04 - val_loss: 5.9763e-04
Epoch 6/30
789/789 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - loss: 7.5380e-04 - val_loss: 6.1792e-04
Epoch 7/30
789/789 ━━━━━━━━━━━━━━━━━━━━ 24s 31ms/step - loss: 7.4604e-04 - val_loss: 5.9943e-04
Epoch 8/30
789/789 ━━━━━━━━━━━━━━━━━━━━ 22s 28ms/step - loss: 7.4661e-04 - val_loss: 5.7934e-04
Epoch 9/30
789/789 ━━━━━━━━━━━━━━━━━━━━ 44s 56ms/step - loss: 7.4535e-04 - val_loss: 5.9046e-04
Epoch 10/30
789/789 ━━━━━━━━━━━━━━━━━━━━ 31s 40ms/step - loss: 7.4175e-04 - val_loss: 6.2283e-04
Epoch 11/30
789/789 ━━━━━━━━━━━━━━━━━━━━ 26

In [ ]:
test_loss = model.evaluate(X_test, y_test, verbose=1)
print("Test Loss:", test_loss)

220/220 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 7.9933e-04
Test Loss: 0.0007993310573510826


In [ ]:
predictions_scaled = model.predict(X_test)
print("Prediction Shape:", predictions_scaled.shape)

220/220 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step
Prediction Shape: (7013, 1)


In [ ]:
predictions = scaler.inverse_transform(predictions_scaled)
actual_values = scaler.inverse_transform(y_test)

In [ ]:
mae = mean_absolute_error(actual_values, predictions)
print(f"MAE: {mae:.2f}")

MAE: 11.29


In [ ]:
mse = mean_squared_error(actual_values, predictions)
print(f"MSE: {mse:.2f}")

MSE: 366.36


In [ ]:
rmse = np.sqrt(mse)
print(f"RMSE: {rmse:.2f}")

RMSE: 19.14


In [ ]:
results = pd.DataFrame(
    {"Actual PM2.5": actual_values.flatten(), "Predicted PM2.5": predictions.flatten()}
)
results["Absolute Error"] = abs(results["Actual PM2.5"] - results["Predicted PM2.5"])
results.head(20)

,Actual PM2.5,Predicted PM2.5,Absolute Error
0,12.0,17.337748,5.337748
1,21.0,18.709467,2.290533
2,28.0,27.868040,0.131960
3,39.0,34.197346,4.802654
4,40.0,45.064022,5.064022
5,47.0,44.913532,2.086468
6,35.0,52.337177,17.337177
7,28.0,38.771042,10.771042
8,25.0,32.976334,7.976334
9,29.0,30.552235,1.552235


In [ ]:
model.save("model.keras")